In [ ]:
import requests
from bs4 import BeautifulSoup

In [ ]:
url = "https://madisonperfumery.com/product-category/perfume/page/2/?per_page=80&la_doing_ajax=true"

try:
    response = requests.get(url)
    response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
    soup = BeautifulSoup(response.text, 'html.parser')
    print("Successfully fetched and parsed the URL. Here's a snippet of the parsed HTML:")
    # Display a snippet of the HTML, for example, the title or first few elements
    if soup.title:
        print(f"Page Title: {soup.title.string}")
    else:
        print("No page title found.")
    # You can add more code here to extract specific data using soup.find(), soup.find_all(), etc.

except requests.exceptions.RequestException as e:
    print(f"Error fetching the URL: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Successfully fetched and parsed the URL. Here's a snippet of the parsed HTML:
Page Title: PERFUME Archives | Page 2 of 9 | Madison Perfumery


In [ ]:
import requests
from bs4 import BeautifulSoup
import json

# # ---------------- INPUT ----------------
# url = "https://madisonperfumery.com/product-category/perfume/eau-de-parfum/"  # change if needed

# headers = {
#     "User-Agent": "Mozilla/5.0"
# }

# # ---------------- REQUEST ----------------
# response = requests.get(url, headers=headers)
# soup = BeautifulSoup(response.text, "html.parser")

products = []

# ---------------- PARSE ----------------
items = soup.select("li.product_item")

for item in items:
    try:
        # Product name + URL
        title_tag = item.select_one("h3.product_item--title a")
        name = title_tag.get_text(strip=True) if title_tag else None
        product_url = title_tag["href"] if title_tag else None

        # Image
        img_tag = item.select_one(".product_item--thumbnail img")
        image_url = img_tag["src"] if img_tag else None

        # Price (handles single or range)
        price_tag = item.select(".price .woocommerce-Price-amount bdi")
        prices = [p.get_text(strip=True) for p in price_tag]
        price = " - ".join(prices) if prices else None

        # Category
        category_tag = item.select_one(".product_item--category-link a")
        category = category_tag.get_text(strip=True) if category_tag else None

        # Description
        desc_tag = item.select_one(".item--excerpt p")
        description = desc_tag.get_text(strip=True) if desc_tag else None

        # Product ID + SKU (from button)
        button = item.select_one(".add_to_cart_button")
        product_id = button.get("data-product_id") if button else None
        sku = button.get("data-product_sku") if button else None

        products.append({
            "name": name,
            "product_url": product_url,
            "image_url": image_url,
            "price": price,
            "category": category,
            "description": description,
            "product_id": product_id,
            "sku": sku
        })

    except Exception as e:
        print("Error parsing item:", e)

# ---------------- OUTPUT ----------------
print(json.dumps(products, indent=2, ensure_ascii=False))

[
  {
    "name": "Yasat EDP WIDIAN",
    "product_url": "https://madisonperfumery.com/madison-onlline/perfume/eau-de-parfum/widian-yasat-edp/",
    "image_url": "https://madisonperfumery.com/wp-content/uploads/Yasat-EDP-WIDIAN-300x300.jpg",
    "price": "1.750lei",
    "category": "EAU DE PARFUM",
    "description": "Inspired by the tranquil embrace of an island oasis, Yasat perfume blends the freshness of…",
    "product_id": "86977",
    "sku": "MX.003083"
  },
  {
    "name": "Alto Astral EDP BYREDO",
    "product_url": "https://madisonperfumery.com/madison-onlline/perfume/eau-de-parfum/alto-astral-edp/",
    "image_url": "https://madisonperfumery.com/wp-content/uploads/ALTO-ASTRAL-41-300x300.jpg",
    "price": "860lei - 1.250lei",
    "category": "EAU DE PARFUM",
    "description": "An embodiment of positive energy, Alto Astral Eau de Parfum takes its name from a…",
    "product_id": "86181",
    "sku": "MX.003041"
  },
  {
    "name": "Max Richter 01 EDT COMME DES GARCONS",
    "

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time

# ---------------- CONFIG ----------------
BASE_URL = "https://madisonperfumery.com/product-category/home"
FIRST_PAGE_URL = f"{BASE_URL}/?per_page=80&la_doing_ajax=true"
PAGE_URL_TEMPLATE = BASE_URL + "/page/{page}/?per_page=80&la_doing_ajax=true"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

OUTPUT_FILE = "home_products.json"

# ---------------- FUNCTION: PARSE PRODUCTS ----------------
def parse_products(html):
    soup = BeautifulSoup(html, "html.parser")
    items = soup.select("li.product_item")

    products = []

    for item in items:
        try:
            title_tag = item.select_one("h3.product_item--title a")
            name = title_tag.get_text(strip=True) if title_tag else None
            product_url = title_tag["href"] if title_tag else None

            img_tag = item.select_one(".product_item--thumbnail img")
            image_url = img_tag["src"] if img_tag else None

            price_tag = item.select(".price .woocommerce-Price-amount bdi")
            prices = [p.get_text(strip=True) for p in price_tag]
            price = " - ".join(prices) if prices else None

            category_tag = item.select_one(".product_item--category-link a")
            category = category_tag.get_text(strip=True) if category_tag else None

            desc_tag = item.select_one(".item--excerpt p")
            description = desc_tag.get_text(strip=True) if desc_tag else None

            button = item.select_one(".add_to_cart_button")
            product_id = button.get("data-product_id") if button else None
            sku = button.get("data-product_sku") if button else None

            products.append({
                "name": name,
                "product_url": product_url,
                "image_url": image_url,
                "price": price,
                "category": category,
                "description": description,
                "product_id": product_id,
                "sku": sku
            })

        except Exception as e:
            print("Error parsing item:", e)

    return products


# ---------------- FUNCTION: FETCH PAGE ----------------
def fetch_page(url):
    try:
        response = requests.get(url, headers=HEADERS, timeout=30)
        response.raise_for_status()
        return response.text
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {url} -> {e}")
        return None


# ---------------- MAIN SCRAPER ----------------
def scrape_all():
    all_products = []
    page = 1

    while True:
        if page == 1:
            url = FIRST_PAGE_URL
        else:
            url = PAGE_URL_TEMPLATE.format(page=page)

        print(f"Fetching page {page}: {url}")

        html = fetch_page(url)
        if not html:
            print("Stopping due to fetch failure.")
            break

        products = parse_products(html)
        count = len(products)

        print(f"Found {count} products on page {page}")

        if count == 0:
            print("No products found. Stopping.")
            break

        all_products.extend(products)

        # ✅ Stop condition
        if count < 80:
            print("Last page reached (less than 80 products).")
            break

        page += 1
        time.sleep(1)  # polite delay

    return all_products


# ---------------- RUN ----------------
if __name__ == "__main__":
    data = scrape_all()

    print(f"\nTotal products scraped: {len(data)}")

    # Save to file
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"Saved to {OUTPUT_FILE}")

Fetching page 1: https://madisonperfumery.com/product-category/home/?per_page=80&la_doing_ajax=true
Found 80 products on page 1
Fetching page 2: https://madisonperfumery.com/product-category/home/page/2/?per_page=80&la_doing_ajax=true
Found 80 products on page 2
Fetching page 3: https://madisonperfumery.com/product-category/home/page/3/?per_page=80&la_doing_ajax=true
Found 80 products on page 3
Fetching page 4: https://madisonperfumery.com/product-category/home/page/4/?per_page=80&la_doing_ajax=true


In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time

# ---------------- CONFIG ----------------
BASE_URL = "https://madisonperfumery.com/product-category"

CATEGORIES = [
    "perfume",
    "make-up",
    "home",
    "bath-body",
    "hair",
    "skin",
    "jewelry",
    "design"
]

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

OUTPUT_FILE = "all_products.json"
DELAY = 1


# ---------------- PARSE ----------------
def parse_products(html, category_name):
    soup = BeautifulSoup(html, "html.parser")
    items = soup.select("li.product_item")

    products = []

    for item in items:
        try:
            title_tag = item.select_one("h3.product_item--title a")
            name = title_tag.get_text(strip=True) if title_tag else None
            product_url = title_tag["href"] if title_tag else None

            img_tag = item.select_one(".product_item--thumbnail img")
            image_url = img_tag["src"] if img_tag else None

            price_tag = item.select(".price .woocommerce-Price-amount bdi")
            prices = [p.get_text(strip=True) for p in price_tag]
            price = " - ".join(prices) if prices else None

            category_tag = item.select_one(".product_item--category-link a")
            sub_category = category_tag.get_text(strip=True) if category_tag else None

            desc_tag = item.select_one(".item--excerpt p")
            description = desc_tag.get_text(strip=True) if desc_tag else None

            button = item.select_one(".add_to_cart_button")
            product_id = button.get("data-product_id") if button else None
            sku = button.get("data-product_sku") if button else None

            products.append({
                "category": category_name,
                "sub_category": sub_category,
                "name": name,
                "product_url": product_url,
                "image_url": image_url,
                "price": price,
                "description": description,
                "product_id": product_id,
                "sku": sku
            })

        except Exception as e:
            print("Error parsing item:", e)

    return products


# ---------------- FETCH ----------------
def fetch_page(url):
    try:
        response = requests.get(url, headers=HEADERS, timeout=30)
        response.raise_for_status()
        return response.text
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {url} -> {e}")
        return None


# ---------------- SCRAPE CATEGORY ----------------
def scrape_category(category):
    print(f"\n========== CATEGORY: {category.upper()} ==========")

    all_products = []
    page = 1

    while True:
        if page == 1:
            url = f"{BASE_URL}/{category}/?per_page=80&la_doing_ajax=true"
        else:
            url = f"{BASE_URL}/{category}/page/{page}/?per_page=80&la_doing_ajax=true"

        print(f"Fetching page {page}: {url}")

        html = fetch_page(url)
        if not html:
            print("Stopping category due to fetch failure.")
            break

        products = parse_products(html, category)
        count = len(products)

        print(f"Found {count} products")

        if count == 0:
            print("No products found. Stopping category.")
            break

        all_products.extend(products)

        # stop condition
        if count < 80:
            print("Reached last page for this category.")
            break

        page += 1
        time.sleep(DELAY)

    return all_products


# ---------------- MAIN ----------------
def scrape_all_categories():
    all_data = []

    for category in CATEGORIES:
        category_data = scrape_category(category)
        all_data.extend(category_data)

    return all_data


# ---------------- RUN ----------------
if __name__ == "__main__":
    data = scrape_all_categories()

    print(f"\nTOTAL PRODUCTS SCRAPED: {len(data)}")

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"Saved to {OUTPUT_FILE}")


========== CATEGORY: PERFUME ==========
Fetching page 1: https://madisonperfumery.com/product-category/perfume/?per_page=80&la_doing_ajax=true
Found 80 products
Fetching page 2: https://madisonperfumery.com/product-category/perfume/page/2/?per_page=80&la_doing_ajax=true
Found 80 products
Fetching page 3: https://madisonperfumery.com/product-category/perfume/page/3/?per_page=80&la_doing_ajax=true
Found 80 products
Fetching page 4: https://madisonperfumery.com/product-category/perfume/page/4/?per_page=80&la_doing_ajax=true
Found 80 products
Fetching page 5: https://madisonperfumery.com/product-category/perfume/page/5/?per_page=80&la_doing_ajax=true
Found 80 products
Fetching page 6: https://madisonperfumery.com/product-category/perfume/page/6/?per_page=80&la_doing_ajax=true
Found 80 products
Fetching page 7: https://madisonperfumery.com/product-category/perfume/page/7/?per_page=80&la_doing_ajax=true
Found 80 products
Fetching page 8: https://madisonperfumery.com/product-category/perfume

In [ ]:
import json

INPUT_FILE = "all_products.json"
OUTPUT_FILE = "all_products_with_brand.json"


# ---------------- SPLIT FUNCTION ----------------
def split_product_brand(title):
    if not title:
        return None, None

    words = title.split()
    brand_words = []

    # capture trailing ALL CAPS words
    for word in reversed(words):
        if word.isupper():
            brand_words.insert(0, word)
        else:
            break

    if brand_words:
        brand = " ".join(brand_words)
        product_name = " ".join(words[:-len(brand_words)])
    else:
        brand = None
        product_name = title

    return product_name.strip(), brand


# ---------------- PROCESS FILE ----------------
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

updated_data = []

for item in data:
    name = item.get("name")

    product_name, brand = split_product_brand(name)

    # add new fields
    item["product_name"] = product_name
    item["brand"] = brand

    updated_data.append(item)


# ---------------- SAVE ----------------
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(updated_data, f, indent=2, ensure_ascii=False)


print(f"Updated file saved as: {OUTPUT_FILE}")

JSONDecodeError: Unterminated string starting at: line 21148 column 20 (char 1043176)

In [ ]:
import json

INPUT_FILE = "all_products.json"

def safe_load_json(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    try:
        return json.loads(content)
    except json.JSONDecodeError as e:
        print("JSON is corrupted. Attempting recovery...")

        # Try trimming the file progressively
        for i in range(len(content), 0, -1000):
            try:
                partial = content[:i]
                data = json.loads(partial + "]")  # try closing array
                print(f"Recovered partial data up to char {i}")
                return data
            except:
                continue

        raise Exception("Could not recover JSON")


data = safe_load_json(INPUT_FILE)
print(f"Loaded {len(data)} valid records")

Loaded 3024 valid records


In [ ]:
def split_product_brand(title):
    if not title:
        return None, None

    words = title.split()
    brand_words = []

    for word in reversed(words):
        if word.isupper():
            brand_words.insert(0, word)
        else:
            break

    if brand_words:
        brand = " ".join(brand_words)
        product_name = " ".join(words[:-len(brand_words)])
    else:
        brand = None
        product_name = title

    return product_name.strip(), brand


cleaned = []

for item in data:
    product_name, brand = split_product_brand(item.get("name"))

    item["product_name"] = product_name
    item["brand"] = brand

    cleaned.append(item)


with open("fixed_products.json", "w", encoding="utf-8") as f:
    json.dump(cleaned, f, indent=2, ensure_ascii=False)

print("Saved cleaned data to fixed_products.json")

Saved cleaned data to fixed_products.json


In [ ]:
import requests
from bs4 import BeautifulSoup

url_to_test = "https://madisonperfumery.com/madison-onlline/perfume/eau-de-parfum/une-nuit-nomade-suma-oriental/"

# Define a User-Agent header to mimic a browser
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

try:
    response = requests.get(url_to_test, headers=HEADERS) # Add headers to the request
    response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
    soup = BeautifulSoup(response.text, 'html.parser')
    print(f"Successfully fetched and parsed: {url_to_test}")
    print("Here's a snippet of the parsed HTML (page title or first paragraph):")
    if soup.title:
        print(f"Page Title: {soup.title.string}")
    elif soup.find('p'):
        print(f"First Paragraph: {soup.find('p').get_text(strip=True)[:100]}...")
    else:
        print("No page title or first paragraph found.")

except requests.exceptions.RequestException as e:
    print(f"Error fetching the URL: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Successfully fetched and parsed: https://madisonperfumery.com/madison-onlline/perfume/eau-de-parfum/une-nuit-nomade-suma-oriental/
Here's a snippet of the parsed HTML (page title or first paragraph):
Page Title: Suma Oriental EDP UNE NUIT NOMADE | Madison Perfumery


In [ ]:
import requests
from bs4 import BeautifulSoup
import os
import json
from urllib.parse import urlparse

# --- CONFIG ---
INPUT_FILE = "/content/fixed_products.json"   # your JSON list
OUTPUT_FOLDER = "html_pages"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}


# --- UTILS ---
def create_folder(folder):
    if not os.path.exists(folder):
        os.makedirs(folder)


def url_to_filename(url):
    """
    Convert URL → clean filename
    Example:
    https://site.com/a/b/c/ → a_b_c.html
    """
    path = urlparse(url).path.strip("/")
    filename = path.replace("/", "_")

    if not filename:
        filename = "homepage"

    return f"{filename}.html"


# --- CORE FETCH FUNCTION ---
def fetch_and_save_html(url):
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.raise_for_status()

        html = response.text
        filename = url_to_filename(url)
        filepath = os.path.join(OUTPUT_FOLDER, filename)

        # Save HTML
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(html)

        print(f"✅ Saved: {filename}")

        return html

    except requests.exceptions.RequestException as e:
        print(f"❌ Failed: {url} | Error: {e}")
        return None


# --- OPTIONAL PARSER (EXTENDABLE) ---
def parse_basic_info(html):
    """
    Minimal parser (you can expand later)
    """
    soup = BeautifulSoup(html, "html.parser")

    data = {}

    # Example extraction
    title = soup.title.string if soup.title else None
    data["page_title"] = title

    return data


# --- MAIN PIPELINE ---
def process_products(input_file):
    create_folder(OUTPUT_FOLDER)

    with open(input_file, "r", encoding="utf-8") as f:
        products = json.load(f)

    results = []

    for i, product in enumerate(products, 1):
        url = product.get("product_url")

        if not url:
            continue

        print(f"\n[{i}/{len(products)}] Fetching: {url}")

        html = fetch_and_save_html(url)

        if not html:
            continue

        # Optional parsing
        parsed_data = parse_basic_info(html)

        # Merge original + parsed
        combined = {**product, **parsed_data}
        results.append(combined)

    return results


# --- RUN ---
if __name__ == "__main__":
    output = process_products(INPUT_FILE)

    # Save enriched results
    with open("enriched_products.json", "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)

    print(f"\n🎯 Done. Processed {len(output)} products.")


[1/3024] Fetching: https://madisonperfumery.com/madison-onlline/perfume/eau-de-parfum/une-nuit-nomade-sun-bleached/
✅ Saved: madison-onlline_perfume_eau-de-parfum_une-nuit-nomade-sun-bleached.html

[2/3024] Fetching: https://madisonperfumery.com/madison-onlline/perfume/eau-de-parfum/une-nuit-nomade-suma-oriental/
✅ Saved: madison-onlline_perfume_eau-de-parfum_une-nuit-nomade-suma-oriental.html

[3/3024] Fetching: https://madisonperfumery.com/madison-onlline/perfume/eau-de-parfum/une-nuit-nomade-sugar-leather-2/
✅ Saved: madison-onlline_perfume_eau-de-parfum_une-nuit-nomade-sugar-leather-2.html

[4/3024] Fetching: https://madisonperfumery.com/madison-onlline/perfume/eau-de-parfum/une-nuit-nomade-spicy-road/
✅ Saved: madison-onlline_perfume_eau-de-parfum_une-nuit-nomade-spicy-road.html

[5/3024] Fetching: https://madisonperfumery.com/madison-onlline/perfume/eau-de-parfum/une-nuit-nomade-nothing-but-sea-and-sky-new/
✅ Saved: madison-onlline_perfume_eau-de-parfum_une-nuit-nomade-nothing-b

In [ ]:
import os
import zipfile
from google.colab import files

folder_to_zip = '/content/html_pages'
output_zip_name = 'html_pages.zip'

print(f"Zipping folder: {folder_to_zip} into {output_zip_name}...")

with zipfile.ZipFile(output_zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_in_dir in os.walk(folder_to_zip):
        for file in files_in_dir:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, folder_to_zip)
            zipf.write(file_path, arcname)

print(f"Successfully created {output_zip_name}")
print("Initiating download...")

files.download(output_zip_name)